In [13]:
import matplotlib
import scipy.io as sio
%matplotlib inline
import seaborn as sns
sns.set()
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold as SKF
from sklearn.linear_model import LogisticRegression as LR
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score as AUC
from sklearn.model_selection import train_test_split
import pylab as plt
from sklearn_pandas import DataFrameMapper
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from pandas.api.types import is_string_dtype, is_numeric_dtype
from sklearn.ensemble import _forest as forest
from sklearn.tree import export_graphviz

In [14]:
#loading test and train data
def load_data(data_folder, filename, Xdata_label, ydata_label):

    total_sections = 5
    X_train_og = np.array([]) 
    y_train_og = np.array([])
    for i in range(total_sections):
        train_fn = filename+str(i+1)+'_'+str(total_sections)+'.mat'
        trainmat = sio.loadmat(data_folder+train_fn)
        if X_train_og.shape[0] == 0:
            X_train_og = np.array(trainmat[Xdata_label])
            y_train_og = np.array(trainmat[ydata_label]).T
        else:
            curr_X_train = np.array(trainmat[Xdata_label])
            curr_y_train = np.array(trainmat[ydata_label]).T
            X_train_og = np.concatenate([X_train_og, curr_X_train], axis=0)
            y_train_og = np.concatenate([y_train_og, curr_y_train], axis=0)
    return X_train_og, y_train_og


In [15]:
data_folder = '/Users/yibeijia/Downloads/nucleosome_occupancy/data/train_test_data/'
train_filename = 'yeastAll_Shapes_Train_5seqsPerClustr_'
X_train_og, y_train_og = load_data(data_folder, train_filename, 'Train_data', 'Train_labels')

In [18]:
test_filename = 'yeastAll_Shapes_Test_5seqsPerClustr_'
X_test_og, y_test_og = load_data(data_folder, test_filename, 'Test_data', 'Test_labels')

In [31]:
X_train_np = np.array(X_train_og)
X_test_np = np.array(X_test_og)

In [48]:
print(X_test_np.shape)

(129525, 146, 54)


In [29]:
Shapes=['Buckle-FL', 'Buckle', 'EP', 'HelT-FL', 'HelT', 'MGW-FL', 'MGW', 
            'Opening-FL', 'Opening', 'ProT-FL', 'ProT', 'Rise-FL', 'Rise', 'Roll-FL',
            'Roll', 'Shear-FL', 'Shear', 'Shift-FL', 'Shift', 'Slide-FL', 'Slide', 
            'Stagger-FL', 'Stagger', 'Stretch-FL', 'Stretch', 'Tilt-FL', 'Tilt']
All_Shapes=[]
for shape in Shapes:
    All_Shapes.append(shape + '-F')
    All_Shapes.append(shape + '-B')
print(len(All_Shapes))
print(All_Shapes)

54
['Buckle-FL-F', 'Buckle-FL-B', 'Buckle-F', 'Buckle-B', 'EP-F', 'EP-B', 'HelT-FL-F', 'HelT-FL-B', 'HelT-F', 'HelT-B', 'MGW-FL-F', 'MGW-FL-B', 'MGW-F', 'MGW-B', 'Opening-FL-F', 'Opening-FL-B', 'Opening-F', 'Opening-B', 'ProT-FL-F', 'ProT-FL-B', 'ProT-F', 'ProT-B', 'Rise-FL-F', 'Rise-FL-B', 'Rise-F', 'Rise-B', 'Roll-FL-F', 'Roll-FL-B', 'Roll-F', 'Roll-B', 'Shear-FL-F', 'Shear-FL-B', 'Shear-F', 'Shear-B', 'Shift-FL-F', 'Shift-FL-B', 'Shift-F', 'Shift-B', 'Slide-FL-F', 'Slide-FL-B', 'Slide-F', 'Slide-B', 'Stagger-FL-F', 'Stagger-FL-B', 'Stagger-F', 'Stagger-B', 'Stretch-FL-F', 'Stretch-FL-B', 'Stretch-F', 'Stretch-B', 'Tilt-FL-F', 'Tilt-FL-B', 'Tilt-F', 'Tilt-B']


In [47]:

for i in X_test_np[0]:
    print(i)
    print(len(i))
# for i in X_train_np:
#     print(i)

[ 6.9274  7.9262 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778
 -1.1665 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778 -1.1665
 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778
 -1.1665 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778 -1.1665
 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778 -1.1665 -0.6778
 -1.1665 -0.6778 -1.1665  6.9274  7.9262 -0.6778 -1.1665 -0.6778 -1.1665]
54
[ 7.9262  7.0861 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665
 -4.3089 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665 -4.3089
 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665
 -4.3089 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665 -4.3089
 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665 -4.3089 -1.1665
 -4.3089 -1.1665 -4.3089  7.9262  7.0861 -1.1665 -4.3089 -1.1665 -4.3089]
54
[ 7.0861  6.9988 -4.3089 -2.7661 -4.3089 -2.7661 -4.3089 -2.7661 -4.3089
 -2.7661 -4.3089 -2.7661 -4.3089 -2.7661 -4

[ 6.7707  7.7649 -1.6508  1.3537 -1.6508  1.3537 -1.6508  1.3537 -1.6508
  1.3537 -1.6508  1.3537 -1.6508  1.3537 -1.6508  1.3537 -1.6508  1.3537
 -1.6508  1.3537 -1.6508  1.3537 -1.6508  1.3537 -1.6508  1.3537 -1.6508
  1.3537 -1.6508  1.3537 -1.6508  1.3537 -1.6508  1.3537 -1.6508  1.3537
 -1.6508  1.3537 -1.6508  1.3537 -1.6508  1.3537 -1.6508  1.3537 -1.6508
  1.3537 -1.6508  1.3537  6.7707  7.7649 -1.6508  1.3537 -1.6508  1.3537]
54
[7.7649 7.4863 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556
 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556
 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556
 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556
 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556 1.3537 1.6556 7.7649 7.4863
 1.3537 1.6556 1.3537 1.6556]
54
[ 7.4863  7.2532  1.6556 -1.9677  1.6556 -1.9677  1.6556 -1.9677  1.6556
 -1.9677  1.6556 -1.9677  1.6556 -1.9677  1.6556 -1.9677  1.6556 -1.9677
  1.6556 -1.9677  1.6556

[ 7.5339  7.438   0.5794 -1.2985  0.5794 -1.2985  0.5794 -1.2985  0.5794
 -1.2985  0.5794 -1.2985  0.5794 -1.2985  0.5794 -1.2985  0.5794 -1.2985
  0.5794 -1.2985  0.5794 -1.2985  0.5794 -1.2985  0.5794 -1.2985  0.5794
 -1.2985  0.5794 -1.2985  0.5794 -1.2985  0.5794 -1.2985  0.5794 -1.2985
  0.5794 -1.2985  0.5794 -1.2985  0.5794 -1.2985  0.5794 -1.2985  0.5794
 -1.2985  0.5794 -1.2985  7.5339  7.438   0.5794 -1.2985  0.5794 -1.2985]
54
[ 7.438   8.2285 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985
 -6.2775 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985 -6.2775
 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985
 -6.2775 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985 -6.2775
 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985 -6.2775 -1.2985
 -6.2775 -1.2985 -6.2775  7.438   8.2285 -1.2985 -6.2775 -1.2985 -6.2775]
54
[ 8.2285  7.7963 -6.2775 -2.5829 -6.2775 -2.5829 -6.2775 -2.5829 -6.2775
 -2.5829 -6.2775 -2.5829 -6.2775 -2.5829 -6

[ 7.5333  7.3359 -0.6454  0.4208 -0.6454  0.4208 -0.6454  0.4208 -0.6454
  0.4208 -0.6454  0.4208 -0.6454  0.4208 -0.6454  0.4208 -0.6454  0.4208
 -0.6454  0.4208 -0.6454  0.4208 -0.6454  0.4208 -0.6454  0.4208 -0.6454
  0.4208 -0.6454  0.4208 -0.6454  0.4208 -0.6454  0.4208 -0.6454  0.4208
 -0.6454  0.4208 -0.6454  0.4208 -0.6454  0.4208 -0.6454  0.4208 -0.6454
  0.4208 -0.6454  0.4208  7.5333  7.3359 -0.6454  0.4208 -0.6454  0.4208]
54
[ 7.3359  6.2218  0.4208 -1.0513  0.4208 -1.0513  0.4208 -1.0513  0.4208
 -1.0513  0.4208 -1.0513  0.4208 -1.0513  0.4208 -1.0513  0.4208 -1.0513
  0.4208 -1.0513  0.4208 -1.0513  0.4208 -1.0513  0.4208 -1.0513  0.4208
 -1.0513  0.4208 -1.0513  0.4208 -1.0513  0.4208 -1.0513  0.4208 -1.0513
  0.4208 -1.0513  0.4208 -1.0513  0.4208 -1.0513  0.4208 -1.0513  0.4208
 -1.0513  0.4208 -1.0513  7.3359  6.2218  0.4208 -1.0513  0.4208 -1.0513]
54
[ 6.2218  7.2514 -1.0513  0.8799 -1.0513  0.8799 -1.0513  0.8799 -1.0513
  0.8799 -1.0513  0.8799 -1.0513  0.8799 -1

[ 8.2749  7.6682 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414
 -0.7337 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414 -0.7337
 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414
 -0.7337 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414 -0.7337
 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414 -0.7337 -3.9414
 -0.7337 -3.9414 -0.7337  8.2749  7.6682 -3.9414 -0.7337 -3.9414 -0.7337]
54
[ 7.6682  7.2614 -0.7337  1.6222 -0.7337  1.6222 -0.7337  1.6222 -0.7337
  1.6222 -0.7337  1.6222 -0.7337  1.6222 -0.7337  1.6222 -0.7337  1.6222
 -0.7337  1.6222 -0.7337  1.6222 -0.7337  1.6222 -0.7337  1.6222 -0.7337
  1.6222 -0.7337  1.6222 -0.7337  1.6222 -0.7337  1.6222 -0.7337  1.6222
 -0.7337  1.6222 -0.7337  1.6222 -0.7337  1.6222 -0.7337  1.6222 -0.7337
  1.6222 -0.7337  1.6222  7.6682  7.2614 -0.7337  1.6222 -0.7337  1.6222]
54
[7.2614 6.5461 1.6222 3.1224 1.6222 3.1224 1.6222 3.1224 1.6222 3.1224
 1.6222 3.1224 1.6222 3.1224 1.6222 3.1224 1.

[9.2207 8.29   0.7239 0.9213 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213
 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213
 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213
 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213
 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213 0.7239 0.9213 9.2207 8.29
 0.7239 0.9213 0.7239 0.9213]
54
[ 8.29    7.6169  0.9213 -0.2077  0.9213 -0.2077  0.9213 -0.2077  0.9213
 -0.2077  0.9213 -0.2077  0.9213 -0.2077  0.9213 -0.2077  0.9213 -0.2077
  0.9213 -0.2077  0.9213 -0.2077  0.9213 -0.2077  0.9213 -0.2077  0.9213
 -0.2077  0.9213 -0.2077  0.9213 -0.2077  0.9213 -0.2077  0.9213 -0.2077
  0.9213 -0.2077  0.9213 -0.2077  0.9213 -0.2077  0.9213 -0.2077  0.9213
 -0.2077  0.9213 -0.2077  8.29    7.6169  0.9213 -0.2077  0.9213 -0.2077]
54
[ 7.6169  8.1526 -0.2077 -1.0317 -0.2077 -1.0317 -0.2077 -1.0317 -0.2077
 -1.0317 -0.2077 -1.0317 -0.2077 -1.0317 -0.2077 -1.0317 -0.2077 -1.0317
 -0.2077 -1.0317 -0.2077 -

[7.6087 7.7617 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402
 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402
 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402
 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402
 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402 0.1407 2.9402 7.6087 7.7617
 0.1407 2.9402 0.1407 2.9402]
54
[ 7.7617  7.6933  2.9402 -2.4113  2.9402 -2.4113  2.9402 -2.4113  2.9402
 -2.4113  2.9402 -2.4113  2.9402 -2.4113  2.9402 -2.4113  2.9402 -2.4113
  2.9402 -2.4113  2.9402 -2.4113  2.9402 -2.4113  2.9402 -2.4113  2.9402
 -2.4113  2.9402 -2.4113  2.9402 -2.4113  2.9402 -2.4113  2.9402 -2.4113
  2.9402 -2.4113  2.9402 -2.4113  2.9402 -2.4113  2.9402 -2.4113  2.9402
 -2.4113  2.9402 -2.4113  7.7617  7.6933  2.9402 -2.4113  2.9402 -2.4113]
54
[ 7.6933  7.4281 -2.4113 -6.611  -2.4113 -6.611  -2.4113 -6.611  -2.4113
 -6.611  -2.4113 -6.611  -2.4113 -6.611  -2.4113 -6.611  -2.4113 -6.611
 -2.4113 -6.611  -2.4113 

In [37]:
A = np.array(range(1,148))
B = np.array(Shapes*len(A))

In [38]:
train_df=pd.DataFrame(data=X_train_np.T, columns=pd.MultiIndex.from_tuples(zip(A,B)))
train_df

ValueError: Must pass 2-d input. shape=(54, 146, 194305)

In [19]:
train_df = pd.DataFrame.from_dict(X_train_og)
test_df = pd.DataFrame.from_dict(X_test_og)

ValueError: Must pass 2-d input. shape=(194305, 146, 54)

In [ ]:
#making copy
trn =  train_df.copy()
tst =  test_df.copy()

In [ ]:
#adding a column to identify whether a row comes from train or not
tst['is_train'] = 0
trn['is_train'] = 1 #1 for train
